##Importando bibliotecas e Carregando imagens

In [5]:
#importando bibliotecas
import cv2 as cv
import numpy as np
import os
from pathlib import Path

In [6]:
def read_img(caminho):
    """Lê e mostra uma imagem."""

    img = cv.imread(caminho)
    if img is not None:
        print("Imagem carregada com sucesso!")
    else: 
        print("Erro ao carregar a imagem.")

def mostrar_img(img):
    cv.imshow("Imagem", img)
    cv.waitKey(0)
    cv.destroyAllWindows()

In [7]:
# Executar as funções
img = read_img("data/raw_images/cast_ok_0_6940.jpeg")

if img is not None:
    mostrar_img(img)

Imagem carregada com sucesso!


In [8]:
def process_batch(input_dir, output_dir, limit=None):
    """Lê todas as imagens do diretório de entrada."""

    # Cria o diretório de saída
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    # Lista somente arquivos de imagem
    files = [f for f in os.listdir(input_dir)if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    # Limita a quantidade de imagens, se necessário
    files = files[:limit]

    if not files:
        print(f"Nenhuma imagem encontrada em {input_dir}.")
        return
    # Processa as imagens
    for file in files:
        caminho = os.path.join(input_dir, file)
        img = cv.imread(caminho)
        if img is not None:
            print(f"Imagem carregada: {file}")
            cv.imshow("Imagem", img)
            cv.waitKey(0)
        else:
            print(f"Erro ao carregar: {file}")
    cv.destroyAllWindows()

In [9]:
# Executar
process_batch("data/raw_images","data/processed_images", limit=5)

Imagem carregada: cast_def_0_0.jpeg
Imagem carregada: cast_def_0_100.jpeg
Imagem carregada: cast_def_0_1015.jpeg
Imagem carregada: cast_def_0_102.jpeg
Imagem carregada: cast_def_0_1046.jpeg


In [10]:
def preprocess_img(img):
    """Aplica o pré-processamento básico na imagem."""

    # Conversão para escala de cinza
    img_cinza = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    # Redução de ruído
    img_gaussian = cv.GaussianBlur(img_cinza, (5, 5), 0)

    return img_cinza, img_gaussian

In [11]:
def segment_image(blur):
    """Aplica threshold Otsu e detecção de bordas."""

    # Limiarização usando Otsu
    _, img_otsu = cv.threshold(blur,0,255,cv.THRESH_BINARY + cv.THRESH_OTSU)

    # Detecção de bordas
    edges = cv.Canny(blur, 50, 150)

    return img_otsu, edges

In [12]:
def morphology_image(img_otsu):
    """Aplica operação morfológica na imagem."""
    kernel = cv.getStructuringElement(cv.MORPH_RECT,(3, 3))
    img_morph = cv.morphologyEx(img_otsu,cv.MORPH_OPEN,kernel)

    return img_morph

def resize_image(img):
    """Redimensiona a imagem para 256x256."""
    img_resize = cv.resize(img, (256, 256))

    return img_resize

In [13]:
img = read_img("data/raw_images/cast_ok_0_6940.jpeg")

if img is not None:

    # Sprint 3
    img_cinza, img_gaussian = preprocess_img(img)

    # Sprint 4
    img_otsu, edges = segment_image(img_gaussian)

    # Sprint 5
    img_morph = morphology_image(img_otsu)
    img_resize = resize_image(img_morph)

    mostrar_img(img_otsu, "Threshold Otsu")
    mostrar_img(img_morph, "Morfologia")
    mostrar_img(img_resize, "Resize 256x256")

Imagem carregada com sucesso!


In [14]:
def process_image(img):
    """Aplica todo o pré-processamento."""
    # Grayscale
    img_cinza = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Gaussian Blur
    img_gaussian = cv.GaussianBlur(img_cinza, (5, 5), 0)
    # Threshold Otsu
    _, img_otsu = cv.threshold(img_gaussian,0,255,cv.THRESH_BINARY + cv.THRESH_OTSU)
    # Morfologia
    kernel = cv.getStructuringElement(cv.MORPH_RECT,(3, 3))
    img_morph = cv.morphologyEx(img_otsu,cv.MORPH_OPEN,kernel)
    # Canny
    edges = cv.Canny(img_gaussian,50,150)
    # Resize
    img_resize = cv.resize(img_morph,(256, 256))

    return img_resize, edges

In [15]:
def process_batch(input_dir, output_dir):
    """Processa e salva todas as imagens."""

    Path(output_dir).mkdir(parents=True,exist_ok=True)

    files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
    if not files:
        print("Nenhuma imagem encontrada.")
        return
    for file in files:

        caminho = os.path.join(input_dir,file)
        img = cv.imread(caminho)
        if img is None:
            print(f"Erro ao carregar: {file}")
            continue
        print(f"Processando: {file}")

        img_processada, edges = process_image(img)
        nome = os.path.splitext(file)[0]
        caminho_saida = os.path.join(output_dir,nome + ".png")
        cv.imwrite(caminho_saida,img_processada)

        print(f"Salva: {caminho_saida}")

In [16]:
#chamando a funcao
process_batch("data/raw_images","data/processed_images")

Processando: cast_def_0_0.jpeg
Salva: data/processed_images\cast_def_0_0.png
Processando: cast_def_0_100.jpeg
Salva: data/processed_images\cast_def_0_100.png
Processando: cast_def_0_1015.jpeg
Salva: data/processed_images\cast_def_0_1015.png
Processando: cast_def_0_102.jpeg
Salva: data/processed_images\cast_def_0_102.png
Processando: cast_def_0_1046.jpeg
Salva: data/processed_images\cast_def_0_1046.png
Processando: cast_def_0_1053.jpeg
Salva: data/processed_images\cast_def_0_1053.png
Processando: cast_def_0_1055.jpeg
Salva: data/processed_images\cast_def_0_1055.png
Processando: cast_def_0_1077.jpeg
Salva: data/processed_images\cast_def_0_1077.png
Processando: cast_def_0_1087.jpeg
Salva: data/processed_images\cast_def_0_1087.png
Processando: cast_def_0_1091.jpeg
Salva: data/processed_images\cast_def_0_1091.png
Processando: cast_def_0_1104.jpeg
Salva: data/processed_images\cast_def_0_1104.png
Processando: cast_def_0_1108.jpeg
Salva: data/processed_images\cast_def_0_1108.png
Processando: c